# Setup

In [25]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
# to load the virtual environment
## & "$HOME\nlp_owlapy_env_py311\Scripts\Activate.ps1"

In [27]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

PROJECT_DIR = Path.cwd()



Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0


In [28]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


torch: 2.11.0+cu128
CUDA available: True
GPU count: 1
GPU 0: NVIDIA RTX A2000 8GB Laptop GPU
Using DEVICE = cuda:0


# Imports

In [29]:
import pickle
import re

import pandas as pd
from spacy.tokens import Doc

from dataclasses import dataclass, asdict
from itertools import product

from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from runtime_config import (
    RUNTIME_PROFILE,
    DEVICE,
    CHUNK_SIZE,
    OVERLAP_SENTENCES,
    MAX_EXPANDED_CHUNK_TOKENS,
)

In [30]:
from coreference.coreference_sub_orchestrator import (
    run_coreference_resolution,
    print_non_singleton_coref_clusters,
)

from coreference.coref_schema import require_coref_layer

In [31]:
from ocean.ocean_probability_scoring import (
    ContextConfig,
    MentionRenderingConfig,
    OceanScoringConfig,
    OceanWeightConfig,
    DirectNLIConfig,
    OceanProbabilityExportConfig,
    export_ocean_probability_csvs,
)

from ocean.ocean_annotator import annotate_doc_with_ocean_folder
from ocean.ocean_schema import (
    require_ocean_layer,
    register_spacy_ocean_extension
)

In [32]:
from cluster_typing.cluster_typing_probability_scoring import (
    ClusterTypingTraversalConfig,
    ClusterTypingScoringConfig,
    ClusterTypingMentionWeightConfig,
    ClusterTypingEvidenceExportConfig,
    export_cluster_typing_evidence_jsonls,
)

from cluster_typing.cluster_typing_annotator import (
    ClusterTypingAnnotationConfig,
    annotate_doc_with_cluster_typing_folder,
)

from cluster_typing.cluster_typing_schema import (
    register_spacy_cluster_typing_extension,
    require_cluster_typing_layer,
)

In [33]:
from ontology.ontology_managment import (
    assert_cluster_relation,
    assert_cluster_type,
    assert_cluster_value,
    load_tbox,
    save_ontology,
    build_relation_catalog,
    build_relation_router,
    # assert_cluster_object_property,
)

from ontology.class_graph import OntologyClassGraph


In [34]:
from relationship_extraction.relation_schema import (
    register_spacy_relation_extension,
    require_relation_layer,
)

from relationship_extraction.extract_relation_candidates import (
    extract_relation_candidates,
    export_routed_relation_candidates_jsonl,
)

from relationship_extraction.align_relation_assignments import (
    RelationNLIConfig,
    load_relation_nli_selector,
    export_relation_assignments_csv,
)

from relationship_extraction.aggregate_cluster_assertions import (
    RelationAggregationConfig,
    export_cluster_assertions_csv,
)

from relationship_extraction.annotate_relation_layer import (
    annotate_relation_layer_from_files,
)

# Functions

In [35]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [36]:
import re

def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Remove standalone page numbers, e.g.
    # This removes only lines that contain digits plus optional whitespace.
    text = re.sub(r"(?m)^[ \t]*\d+[ \t]*$", "", text)

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [37]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str,
    local_path: str | Path,
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [38]:
def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # BookNLP/tokenizer extensions must be registered before unpickling the Doc.
    # The global coreference extension is registered inside the coreference
    # sub-orchestrator immediately before final annotation.
    ensure_booknlp_extensions()
    register_spacy_cluster_typing_extension()
    register_spacy_ocean_extension()
    register_spacy_relation_extension()
    
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


# Config

## I/O config

In [ ]:
# Requested local file
TEXT_FILENAME = "cosmicomics_a_sign_in_space.txt"
LOCAL_TEXT_PATH = Path(f"./resources/books/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / f"outputs_[{TEXT_FILENAME}]"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ONTOLOGY_ROOT = PROJECT_DIR / "resources/ontologies"

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
COREF_DOC_PATH = OUTPUT_ROOT / "coref_doc.pkl"
ONTOLOGY_TYPED_DOC_PATH = OUTPUT_ROOT / "ontology_typed_doc.pkl"
OCEAN_SCORED_DOC_PATH = OUTPUT_ROOT / "ocean_scored_doc.pkl"

ONTOLOGY_TTL_PATH = ONTOLOGY_ROOT/ "narrative_entity_ontology_clean.ttl"

RELATION_OUTPUT_DIR = OUTPUT_ROOT / "relations"
RELATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROUTED_RELATION_CANDIDATES_PATH = RELATION_OUTPUT_DIR / "routed_relation_candidates.jsonl"
RELATION_ASSIGNMENTS_PATH = RELATION_OUTPUT_DIR / "relation_assignments.csv"
CLUSTER_ASSERTIONS_PATH = RELATION_OUTPUT_DIR / "cluster_assertions.csv"
RELATION_DOC_PATH = OUTPUT_ROOT / "relation_doc.pkl"
POPULATED_ONTOLOGY_PATH = OUTPUT_ROOT / "populated_ontology.ttl"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("COREF_DOC_PATH =", COREF_DOC_PATH)
print("ONTOLOGY_TYPED_DOC_PATH =", ONTOLOGY_TYPED_DOC_PATH)
print("OCEAN_SCORED_DOC_PATH =", OCEAN_SCORED_DOC_PATH)
print("ONTOLOGY_TTL_PATH =", ONTOLOGY_TTL_PATH)


TEXT_PATH = resources\books\the_canterville_ghost.txt
OUTPUT_ROOT = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]
TOKENIZED_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\tokenized_doc.pkl
COREF_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\coref_doc.pkl
ONTOLOGY_TYPED_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\ontology_typed_doc.pkl
OCEAN_SCORED_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\ocean_scored_doc.pkl
ONTOLOGY_TTL_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\resources\ontologies\narrative_entity_ontology_clean.ttl


In [40]:
GLOBAL_COREF_OUTPUT_DIR = OUTPUT_ROOT / "global_coref"
GLOBAL_COREF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)

GLOBAL_COREF_OUTPUT_DIR = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\global_coref


## Runtime and pipeline config

In [41]:
# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

RELATION_PAIR_BATCH_SIZE = 64
RELATION_OVERWRITE_STAGE_1 = False
RELATION_OVERWRITE_STAGE_2 = False
RELATION_RESUME_STAGE_2 = True
RELATION_OVERWRITE_STAGE_3 = True

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)


RUNTIME_PROFILE = {'kind': 'local_cuda', 'env': 'Local', 'cuda_available': True, 'gpu_count': 1, 'gpu_name': 'NVIDIA RTX A2000 8GB Laptop GPU', 'device': 'cuda:0', 'cpu_load_first': True, 'precision_policy': 'auto', 'p100_fallback_to_float32': False, 'default_chunk_size': 6000, 'default_overlap_sentences': 60}
DEVICE = cuda:0
CHUNK_SIZE(expanded) = 6000
OVERLAP_SENTENCES = 60
MAX_EXPANDED_CHUNK_TOKENS = 6000
GLOBAL_COREF_OUTPUT_DIR = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\global_coref


# Main

### Tokenization

In [42]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

[doc] Loading tokenized Doc from c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\tokenized_doc.pkl
Doc tokens: 13552
Doc sentences: 401


### Chunking

In [43]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

### Ontology building


In [44]:
onto, ontology_graph = load_tbox(ONTOLOGY_TTL_PATH, require_property_descriptions=True)
ontology_class_graph = OntologyClassGraph(ontology_graph)

print(ontology_graph.number_of_nodes(), "classes")
print(ontology_graph.number_of_edges(), "subclass edges")

relation_catalog = build_relation_catalog(
    onto=onto,
    ontology_path=ONTOLOGY_TTL_PATH,
)

relation_router = build_relation_router(
    class_graph=ontology_graph,
    relation_catalog=relation_catalog,
)

print("Object properties:", len(relation_catalog))


23 classes
22 subclass edges
Object properties: 16


## Node extraction

### Coreference clusters extraction

In [45]:
if COREF_DOC_PATH.exists():
    print(f"[coref-annotation] Loading coref-annotated Doc from {COREF_DOC_PATH}")
    doc = load_doc(COREF_DOC_PATH)
else:
    print("[coref-annotation] Annotated Doc not found; calling annotator sub-orchestrator.")
    doc = run_coreference_resolution(
        doc=doc,
        chunker=chunker,
        chunk_plan=chunk_plan,
        document_id=TEXT_PATH.stem,
        output_dir=GLOBAL_COREF_OUTPUT_DIR,
    )
    save_doc(doc, COREF_DOC_PATH)
    print("[coref-annotation] Saved annotated Doc to:", COREF_DOC_PATH)
    
    
if not Doc.has_extension("coref_layer"):
    Doc.set_extension("coref_layer", default=None)

[coref-annotation] Loading coref-annotated Doc from c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\coref_doc.pkl


In [46]:
print("[coref-annotation] Summary:")
print(doc._.coref_layer.summary())
print_non_singleton_coref_clusters(doc)

[coref-annotation] Summary:
{'n_mentions': 1482, 'n_clusters': 15, 'n_indexed_tokens': 1962, 'n_indexed_heads': 1347, 'n_indexed_spans': 1354}
Non-singleton clusters: 15

cluster_id=3 | canonical_name='the ghost' | n_mentions=450

cluster_id=6 | canonical_name='Virginia' | n_mentions=255

cluster_id=0 | canonical_name='Mr. Otis' | n_mentions=208

cluster_id=2 | canonical_name='Lord Canterville' | n_mentions=119

cluster_id=4 | canonical_name='my family' | n_mentions=70

cluster_id=14 | canonical_name='Cecil' | n_mentions=68

cluster_id=8 | canonical_name='the twins' | n_mentions=65

cluster_id=5 | canonical_name='Mrs. Otis' | n_mentions=59

cluster_id=1 | canonical_name='the house' | n_mentions=33

cluster_id=10 | canonical_name='his room' | n_mentions=32

cluster_id=13 | canonical_name='the group' | n_mentions=30

cluster_id=9 | canonical_name='Mrs. Umney' | n_mentions=29

cluster_id=11 | canonical_name='the little Duke' | n_mentions=27

cluster_id=12 | canonical_name='a band of gipsi

### Ontology cluster typing

In [47]:
ontology_n_mentions = None  # None = type ALL mentions for each requested cluster
ontology_random_seed = 67

In [48]:
if ONTOLOGY_TYPED_DOC_PATH.exists():
    print(f"[ontology typing] Loading ontology-typed Doc from {ONTOLOGY_TYPED_DOC_PATH}")
    doc = load_doc(ONTOLOGY_TYPED_DOC_PATH)
else:
    if not ONTOLOGY_TTL_PATH.exists():
        raise FileNotFoundError(
            f"Ontology TTL not found: {ONTOLOGY_TTL_PATH}. "
            "Create a networkx.DiGraph by any external method, or place the TTL here."
        )

    jsonl_paths_by_cluster_id = export_cluster_typing_evidence_jsonls(
        doc,
        ontology_graph,
        ClusterTypingEvidenceExportConfig(
            n_mentions_per_cluster=ontology_n_mentions,
            output_root=OUTPUT_ROOT,

            context_config=ContextConfig(
                n_sentences_before=0,
                n_sentences_after=0,
                mark_mention=True,
                deduplicate=False,
            ),
            rendering_config=MentionRenderingConfig(
                canonicalize_simple_mentions=True,
                keep_first_second_person=True,
            ),
            traversal_config=ClusterTypingTraversalConfig(
                skip_single_root=True,
                include_stay_option=True,
                force_leaf=False,
            ),
            scoring_config=ClusterTypingScoringConfig(
                subject_aware=True,
            ),
            mention_weight_config=ClusterTypingMentionWeightConfig(),
            nli_config=DirectNLIConfig(
                pair_batch_size=64,
            ),

            chunk_size=16,

            overwrite_jsonl=True,
            resume_from_jsonl=False,

            random_seed=ontology_random_seed,
            sort_sample_by_cluster_order=True,
            print_progress=True,
        ),
    )

    print(jsonl_paths_by_cluster_id)

    n_part = "all" if ontology_n_mentions is None else str(ontology_n_mentions)
    ontology_folder = OUTPUT_ROOT / "cluster_typing" / n_part

    doc = annotate_doc_with_cluster_typing_folder(
        doc,
        ontology_class_graph,
        ontology_folder,
        config=ClusterTypingAnnotationConfig(
            use_mention_weight=True,
        ),
    )
    save_doc(doc, ONTOLOGY_TYPED_DOC_PATH)
    print("\n\n\n[ontology typing] Saved annotated Doc to:", ONTOLOGY_TYPED_DOC_PATH)

cluster_typing_layer = require_cluster_typing_layer(doc)
print(cluster_typing_layer.summary())

All-cluster cluster-typing evidence export
cluster source: doc._.coref_layer.clusters
n_clusters: 15
cluster_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
n_mentions_per_cluster: None
output_dir: c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\cluster_typing\all

----------------------------------------------------------------------------------------------------
cluster 1/15
cluster_id: 0
subject: 'Mr. Otis'
jsonl_path: c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\cluster_typing\all\cluster_typing_evidence_cluster_0_Mr._Otis_all.jsonl
per_cluster_seed: 12622576784306766662
----------------------------------------------------------------------------------------------------
Cluster-typing evidence JSONL export
cluster_id: 0
subject: 'Mr. Otis'
requested mentions: None
total mentions in cluster: 208
jsonl_path: c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outpu

c:\Users\Bobby\nlp_owlapy_env_py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 394/394 [00:00<00:00, 7964.97it/s]


[chunk 1/13] mentions=0:16 | chunk_time=16.27s | done_total=16/208
[chunk 2/13] mentions=16:32 | chunk_time=9.45s | done_total=32/208


KeyboardInterrupt: 

In [ ]:
coref = doc._.coref_layer
cluster_typing_layer = doc._.cluster_typing_layer

rows = []

for cluster_id, cluster in sorted(coref.clusters.items()):
    rows.append(
        {
            "cluster_id": cluster_id,
            "canonical_name": cluster.canonical_name,
            "semantic_type": cluster.semantic_type,
            "n_mentions": len(cluster.mention_ids),
            "class_iri": cluster_typing_layer.class_iri(cluster_id),
            "ontology_class_label": cluster_typing_layer.class_label(cluster_id),
            "ontology_class_human_readable_label": cluster_typing_layer.class_human_readable_label(cluster_id),
        }
    )

ontology_clusters_df = pd.DataFrame(rows)

ontology_clusters_df


### OCEAN scoring

In [ ]:
def cluster_ids_of_class_or_subclass(doc, coref_layer, cluster_typing_layer, class_label):
    coref_cluster_ids = set(coref_layer.clusters)

    return [
        cluster_id
        for cluster_id in cluster_typing_layer.cluster_ids_under(class_label)
        if cluster_id in coref_cluster_ids
    ]

In [ ]:

coref = doc._.coref_layer
cluster_typing_layer = doc._.cluster_typing_layer

cluster_ids = cluster_ids_of_class_or_subclass(
    doc,
    coref,
    cluster_typing_layer,
    "Character",
)

for id in sorted(cluster_ids):
    print(id, end=', ')

0, 2, 3, 5, 6, 7, 8, 9, 11, 14, 

In [ ]:
n_mentions = None  # None = score ALL mentions for each requested cluster
random_seed = 67

In [ ]:
if OCEAN_SCORED_DOC_PATH.exists():
    print(f"[OCEAN scoring] Loading ocean_scoring-annotated Doc from {OCEAN_SCORED_DOC_PATH}")
    doc = load_doc(OCEAN_SCORED_DOC_PATH)
else:

    csv_paths_by_cluster_id = export_ocean_probability_csvs(
    doc,
    OceanProbabilityExportConfig(
        cluster_ids=cluster_ids,
        n_mentions_per_cluster=n_mentions,
        output_root=OUTPUT_ROOT,

        context_config=ContextConfig(
            n_sentences_before=0,
            n_sentences_after=0,
            mark_mention=True,
            deduplicate=False,
        ),
        rendering_config=MentionRenderingConfig(
            canonicalize_simple_mentions=True,
            keep_first_second_person=True,
        ),
        scoring_config=OceanScoringConfig(
            subject_aware=True,
        ),
        weight_config=OceanWeightConfig(),
        nli_config=DirectNLIConfig(
            pair_batch_size=64, # on stronger hardware can be safely set to 64
        ),

        chunk_size=64,

        overwrite_csv=False,
        resume_from_csv=True,
        use_sqlite_cache=True,

        random_seed=random_seed,
        sort_sample_by_cluster_order=True,
        return_dataframes=False,
        print_progress=True,
    ),
)


    print(csv_paths_by_cluster_id)


    n_part = "all" if n_mentions is None else str(n_mentions)
    ocean_folder = OUTPUT_ROOT / "OCEAN_profiles" / n_part

    doc = annotate_doc_with_ocean_folder(
        doc,
        ocean_folder,
    )
    save_doc(doc, OCEAN_SCORED_DOC_PATH)
    print("\n\n\n[OCEAN scoring] Saved annotated Doc to:", OCEAN_SCORED_DOC_PATH)

ocean = require_ocean_layer(doc)
print(ocean.summary())
#### test

Multi-cluster OCEAN probability export
cluster_ids: [0, 11, 14, 2, 3, 5, 6, 7, 8, 9]
n_mentions_per_cluster: None
output_dir: c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\OCEAN_profiles\all

----------------------------------------------------------------------------------------------------
cluster 1/10
cluster_id: 0
subject: 'Mr. Otis'
csv_path: c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\OCEAN_profiles\all\OCEAN_scores_cluster_0_Mr._Otis_all.csv
per_cluster_seed: 12622576784306766662
----------------------------------------------------------------------------------------------------
OCEAN probability CSV export
cluster_id: 0
subject: 'Mr. Otis'
requested mentions: None
total mentions in cluster: 208
csv_path: c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\OCEAN_profiles\all\OCEAN_scores_cluster_0_Mr._Otis_all.csv
cache_path: c:\

In [ ]:
coref = doc._.coref_layer
ocean = doc._.ocean_layer

for cluster_id in cluster_ids:
    print(coref.clusters[cluster_id].canonical_name, sep="->")
    print(ocean.cluster(cluster_id).scores.as_dict())

Mr. Otis
{'openness': 52.8, 'conscientiousness': 59.1, 'extraversion': 54.93, 'agreeableness': 59.13, 'neuroticism': 40.62}
the little Duke
{'openness': 46.02, 'conscientiousness': 49.43, 'extraversion': 61.78, 'agreeableness': 68.37, 'neuroticism': 70.74}
Cecil
{'openness': 45.56, 'conscientiousness': 59.58, 'extraversion': 57.11, 'agreeableness': 69.21, 'neuroticism': 40.65}
Lord Canterville
{'openness': 47.44, 'conscientiousness': 60.91, 'extraversion': 52.69, 'agreeableness': 62.07, 'neuroticism': 45.18}
the ghost
{'openness': 50.41, 'conscientiousness': 54.01, 'extraversion': 42.61, 'agreeableness': 43.05, 'neuroticism': 57.88}
Mrs. Otis
{'openness': 52.42, 'conscientiousness': 57.76, 'extraversion': 57.41, 'agreeableness': 64.58, 'neuroticism': 60.35}
Virginia
{'openness': 50.86, 'conscientiousness': 52.37, 'extraversion': 53.11, 'agreeableness': 61.5, 'neuroticism': 47.17}
the Dukes of Cheshire
{'openness': 47.67, 'conscientiousness': 50.54, 'extraversion': 62.8, 'agreeableness'

## Edge extraction

In [ ]:
for relation in relation_catalog:
    print(relation)

https://example.org/nlp-sdai/narrative-entity-ontology#usesObject
https://example.org/nlp-sdai/narrative-entity-ontology#participatesIn
https://example.org/nlp-sdai/narrative-entity-ontology#occursAt
https://example.org/nlp-sdai/narrative-entity-ontology#objectUsedBy
https://example.org/nlp-sdai/narrative-entity-ontology#movesToward
https://example.org/nlp-sdai/narrative-entity-ontology#memberOf
https://example.org/nlp-sdai/narrative-entity-ontology#locatedIn
https://example.org/nlp-sdai/narrative-entity-ontology#hostsEvent
https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward
https://example.org/nlp-sdai/narrative-entity-ontology#hasParticipant
https://example.org/nlp-sdai/narrative-entity-ontology#hasMember
https://example.org/nlp-sdai/narrative-entity-ontology#harms
https://example.org/nlp-sdai/narrative-entity-ontology#guides
https://example.org/nlp-sdai/narrative-entity-ontology#containsPlace
https://example.org/nlp-sdai/narrative-entity-ontology#containedInPl

In [ ]:
def print_tree(g, root=None, indent=""):
    def label(n):
        data = g.nodes[n]
        return data.get("human_readable_label") or data.get("label") or str(n).split("#")[-1]

    if root is None:
        roots = [n for n in g.nodes if g.in_degree(n) == 0]
        for r in roots:
            print_tree(g, r)
        return

    print(indent + label(root))

    children = sorted(g.successors(root), key=label)
    for child in children:
        print_tree(g, child, indent + "    ")

print_tree(ontology_graph)

Narrative Entity
    Character
        Human Character
        Non Human Character
    Concept
        Belief Or Value
        Power Or Ability
        Rule Or Law
    Event
        Action
        Occurrence
    Group
        Informal Group
        Organization
        People Or Species
    Object
        Document
        Item
        Substance
    Place
        Region
        Settlement
        Site


In [ ]:
# Inspect relation_router

import pandas as pd

def short(iri):
    iri = str(iri)
    return iri.rsplit("#", 1)[-1] if "#" in iri else iri.rstrip("/").rsplit("/", 1)[-1]

catalog = relation_router.relation_catalog

router_df = pd.DataFrame([
    {
        "property": short(spec.iri),
        "domains": ", ".join(short(x) for x in spec.domains),
        "ranges": ", ".join(short(x) for x in spec.ranges),
        "label": spec.human_readable_label,
        "description": spec.description,
    }
    for spec in catalog.values()
])

print(f"Object properties in router: {len(router_df)}")
display(router_df.sort_values("property"))

Object properties in router: 16


,property,domains,ranges,label,description
15,accompanies,"Thing, Character, NarrativeEntity","Thing, Character, NarrativeEntity",Accompanies,Connects a character to another character with...
14,containedInPlace,"Thing, NarrativeEntity, Place","Thing, NarrativeEntity, Place",Contained In Place,Connects a place to a larger place that contai...
13,containsPlace,"Thing, NarrativeEntity, Place","Thing, NarrativeEntity, Place",Contains Place,Connects a place to a smaller place contained ...
12,guides,"Thing, Character, NarrativeEntity","Thing, NarrativeEntity",Guides,Connects a character to an entity that the cha...
11,harms,"Thing, Character, NarrativeEntity","Thing, NarrativeEntity",Harms,Connects a character to an entity that the cha...
10,hasMember,"Thing, Group, NarrativeEntity","Thing, NarrativeEntity",Has Member,Connects a group to one of its members.
9,hasParticipant,"Thing, Event, NarrativeEntity","Thing, NarrativeEntity",Has Participant,Connects an event to a narrative entity that p...
8,holdsAttitudeToward,"Thing, Character, NarrativeEntity","Thing, NarrativeEntity",Holds Attitude Toward,Connects a character to an entity toward which...
7,hostsEvent,"Thing, NarrativeEntity, Place","Thing, Event, NarrativeEntity",Hosts Event,Connects a place to an event that occurs there.
6,locatedIn,"Thing, NarrativeEntity","Thing, NarrativeEntity, Place",Located In,Connects a narrative entity to a place where i...


In [ ]:
relation_candidates_df = extract_relation_candidates(
    doc,
    cluster_typing_layer=doc._.cluster_typing_layer,
)

relation_candidates_df[
    [
        "source_cluster_id",
        "source_canonical_name",
        "source_class_iri",
        "predicate",
        "target_cluster_id",
        "target_canonical_name",
        "target_class_iri",
        "sentence_text",
        "is_passive",
        "is_negated",
    ]
]

export_routed_relation_candidates_jsonl(
    doc=doc,
    cluster_typing_layer=doc._.cluster_typing_layer,
    relation_router=relation_router,
    output_path=ROUTED_RELATION_CANDIDATES_PATH,
    print_discards=True,
    overwrite=True,
)


In [ ]:
selector = load_relation_nli_selector(
    RelationNLIConfig(
        pair_batch_size=RELATION_PAIR_BATCH_SIZE,
    )
)

export_relation_assignments_csv(
    input_path=ROUTED_RELATION_CANDIDATES_PATH,
    output_path=RELATION_ASSIGNMENTS_PATH,
    selector=selector,
    overwrite=RELATION_OVERWRITE_STAGE_2,
    resume=RELATION_RESUME_STAGE_2,
)

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 9089.07it/s]


[relation alignment] Wrote 83 assignments to c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\relations\relation_assignments.csv (0 skipped by resume).


WindowsPath('c:/Users/Bobby/Desktop/real_shit/NLP_character_graph_pipeline/outputs_[the_canterville_ghost.txt]/relations/relation_assignments.csv')

In [ ]:
export_cluster_assertions_csv(
    assignments_path=RELATION_ASSIGNMENTS_PATH,
    output_path=CLUSTER_ASSERTIONS_PATH,
    aggregation_config=RelationAggregationConfig(
        aggregation_method="sum_softmax_by_cluster_pair",
        min_support_count=1,
        min_score=0.0,
    ),
    overwrite=RELATION_OVERWRITE_STAGE_3,
)

[relation aggregation] Wrote 39 cluster assertions to c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline\outputs_[the_canterville_ghost.txt]\relations\cluster_assertions.csv


WindowsPath('c:/Users/Bobby/Desktop/real_shit/NLP_character_graph_pipeline/outputs_[the_canterville_ghost.txt]/relations/cluster_assertions.csv')

In [ ]:
relation_layer = annotate_relation_layer_from_files(
    doc=doc,
    assignments_path=RELATION_ASSIGNMENTS_PATH,
    cluster_assertions_path=CLUSTER_ASSERTIONS_PATH,
    force=True,
)

print(relation_layer.summary())

{'n_relation_assignments': 83, 'n_cluster_assertions': 39, 'n_indexed_predicate_tokens': 61, 'n_indexed_source_clusters': 11, 'n_indexed_target_clusters': 14, 'n_indexed_assertion_source_clusters': 11, 'n_indexed_assertion_target_clusters': 14, 'n_indexed_object_properties': 5}


In [ ]:
coref = require_coref_layer(doc)
relations = require_relation_layer(doc)

for assignment_id in list(relations.assignments):
    source = relations.source_cluster(coref, assignment_id)
    target = relations.target_cluster(coref, assignment_id)
    predicate = relations.predicate_span(doc, assignment_id)
    prop = relations.chosen_object_property_iri(assignment_id)
    conf = relations.confidence(assignment_id)

    print(
        f"{source.canonical_name} -- {predicate.text} / {prop} "
        f"({conf:.3f}) --> {target.canonical_name}"
    )


Mr. Otis -- bought / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.281) --> the house
Mr. Otis -- mind / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.302) --> the ghost
Lord Canterville -- warned / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.428) --> Mr. Otis
Mr. Otis -- went down to / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.316) --> the house
my family -- reached / https://example.org/nlp-sdai/narrative-entity-ontology#hasMember (0.555) --> the house
Mrs. Umney -- waited on / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.344) --> my family
Mrs. Otis -- replied / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.266) --> Mrs. Umney
Mr. Otis -- looked round at / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.344) --> my family
Mrs. Umney -- warned / https://examp

In [ ]:
save_doc(doc, RELATION_DOC_PATH)

WindowsPath('c:/Users/Bobby/Desktop/real_shit/NLP_character_graph_pipeline/outputs_[the_canterville_ghost.txt]/relation_doc.pkl')

In [ ]:
def _local_name(iri: str) -> str:
    """Compact IRI rendering: http://x/y#livesIn -> livesIn."""
    return str(iri).rstrip("/#").rsplit("#", 1)[-1].rsplit("/", 1)[-1]


def _cluster_name(coref_layer, cluster_id: int) -> str:
    """
    Robust cluster label getter.
    Works even if your Cluster class uses slightly different field names.
    """
    cluster = coref_layer.clusters[int(cluster_id)]

    for attr in (
        "canonical_name",
        "canonical_text",
        "representative_text",
        "main_text",
        "name",
        "label",
    ):
        value = getattr(cluster, attr, None)
        if value:
            return str(value)

    # Fallback: try first mention span-like object if available.
    mentions = getattr(cluster, "mentions", None)
    if mentions:
        first = mentions[0]
        text = getattr(first, "text", None)
        if text:
            return str(text)

    return f"cluster_{cluster_id}"


relation_layer = require_relation_layer(doc)
coref_layer = require_coref_layer(doc)

relation_layer.rebuild_indexes()

assertions = sorted(
    relation_layer.cluster_assertions.items(),
    key=lambda item: (
        int(item[1].source_cluster_id),
        _local_name(item[1].object_property_iri),
        int(item[1].target_cluster_id),
    ),
)

cluster_typing_layer = doc._.cluster_typing_layer
print("=" * 100)
print(f"RELATIONSHIP CLUSTERS: {len(assertions)}")
print("=" * 100)

for i, (assertion_id, assertion) in enumerate(assertions, start=1):
    source_id = int(assertion.source_cluster_id)
    target_id = int(assertion.target_cluster_id)

    source_name = _cluster_name(coref_layer, source_id)
    target_name = _cluster_name(coref_layer, target_id)
    prop_name = _local_name(assertion.object_property_iri)

    source_type = cluster_typing_layer.class_human_readable_label(source_id)
    target_type = cluster_typing_layer.class_human_readable_label(target_id)

    support_count = relation_layer.support_count(assertion_id)
    mean_conf = relation_layer.mean_assertion_confidence(assertion_id)

    print(f"\n{source_name}({source_type}) --[{prop_name}]--> {target_name}({target_type})")

RELATIONSHIP CLUSTERS: 39

Mr. Otis(Human Character) --[holdsAttitudeToward]--> the house(Site)

Mr. Otis(Human Character) --[holdsAttitudeToward]--> Lord Canterville(Human Character)

Mr. Otis(Human Character) --[holdsAttitudeToward]--> the ghost(Non Human Character)

Mr. Otis(Human Character) --[holdsAttitudeToward]--> my family(People Or Species)

Mr. Otis(Human Character) --[holdsAttitudeToward]--> Mrs. Otis(Human Character)

Mr. Otis(Human Character) --[holdsAttitudeToward]--> Virginia(Human Character)

Mr. Otis(Human Character) --[holdsAttitudeToward]--> Mrs. Umney(Human Character)

Mr. Otis(Human Character) --[holdsAttitudeToward]--> the little Duke(Human Character)

Mr. Otis(Human Character) --[holdsAttitudeToward]--> Cecil(Human Character)

Mr. Otis(Human Character) --[movesToward]--> his room(Site)

Mr. Otis(Human Character) --[movesToward]--> a band of gipsies(People Or Species)

Mr. Otis(Human Character) --[movesToward]--> the group(People Or Species)

Lord Canterville(Huma